# Conversion of TARGET ODI SQL File

**Conversion Timestamp:** 2024-07-30T12:00:00Z

**Description:** This notebook contains the Spark SQL conversion of an ODI process to insert data into the `trg_dep` table from the `departments` table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type (e.g., F for Full, I for Incremental)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  '${DATASOURCE_NUM_ID}' AS datasource_num_id,
  '${ETL_PROC_WID}' AS etl_proc_wid,
  '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Insert into Target Table

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
INSERT INTO workspace.hr.trg_dep
  (
    department_id ,
    department_name ,
    manager_id ,
    location_id 
  ) 
SELECT 
  departments.department_id ,
  departments.department_name ,
  departments.manager_id ,
  departments.location_id  
FROM 
  workspace.hr.departments departments;

## Validation

In [ ]:
%sql
SELECT COUNT(*) FROM workspace.hr.trg_dep;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names**: Oracle schema `HR` has been mapped to `workspace.hr`. Oracle table names `TRG_DEP` and `DEPARTMENTS` have been converted to lowercase `trg_dep` and `departments` respectively, prefixed with `workspace.hr`.
2.  **Oracle Hints**: The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable to Databricks Delta Lake.
3.  **Data Types**: The DML statement does not explicitly define data types. It is assumed that the target table `workspace.hr.trg_dep` has been created with appropriate Spark SQL data types (e.g., `BIGINT` for ID columns, `STRING` for name columns, `BIGINT` for manager/location IDs).
4.  **SCEN_TASK_NO**: The `SCEN_TASK_NO` markers have been preserved as comments within the relevant SQL cell.
5.  **Parameters**: Standard ETL parameters are created as `dbutils` widgets, though not directly used in this specific SQL statement.